# Compare Qwen VLM Predictions with Manual Annotations

Select model and phrase to compare Qwen predictions with manual annotations.

In [7]:
import pandas as pd
from pathlib import Path

# Configure pandas to display full content without truncation
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [8]:
# Paths
BASE_DIR = Path("/mnt/lustre/work/hein/hof217/stable_diffusion")
MANUAL_DIR = BASE_DIR / "z5-manual_annotations"
QWEN_BASE_DIR = BASE_DIR / "x7-z1-qwen_small/new_qwen_outputs"

# Model mapping
MODELS = ['redcaps', 'cogvlm', 'standard']
MODEL_FILES = {
    'redcaps': 'redcaps_clic.csv',
    'cogvlm': 'cogvlm_clic.csv',
    'standard': 'standard.csv'
}
STRATEGIES = ['direct', 'expert', 'inverse']

In [9]:
def parse_indices(s):
    """Parse semicolon-separated indices."""
    if pd.isna(s) or str(s).strip() == '':
        return set()
    return set(int(x.strip()) for x in str(s).replace(';', ',').split(',') if x.strip().isdigit())

def normalize_answer(ans):
    """Normalize Qwen answer to Yes/No/Ambiguous."""
    if pd.isna(ans) or str(ans).strip() == '':
        return 'Ambiguous'
    # Strip whitespace and remove trailing punctuation (., !, ?, etc.)
    ans = str(ans).strip().lower().rstrip('.,!?;:')
    return 'Yes' if ans == 'yes' else 'No' if ans == 'no' else 'Ambiguous'

def get_qwen_answer(row):
    """Extract Qwen answer from row.
    Checks predicted_label first, falls back to full_output if empty.
    This handles 'direct' strategy where answer is in full_output as 'Yes', 'Yes.', 'No', 'No.'.
    """
    # First try predicted_label
    pred_label = row.get('predicted_label', '')
    if pd.notna(pred_label) and str(pred_label).strip() != '':
        return normalize_answer(pred_label)
    # If predicted_label is empty, use full_output
    full_output = row.get('full_output', '')
    return normalize_answer(full_output)

def get_manual_label(idx, data):
    """Get manual label for tensor index."""
    if data is None or idx >= data['checked']:
        return None
    if idx in data['incorrect']:
        return 'No'
    if idx in data['ambiguous']:
        return 'Ambiguous'
    return 'Yes'

def load_manual_annotations(model, phrase):
    """Load manual annotations for model and phrase."""
    df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
    row = df[df['phrase'] == phrase]
    if row.empty:
        return None
    row = row.iloc[0]
    if pd.isna(row['checked']):
        return None
    return {
        'tensor_file': row['tensor_file'],
        'ambiguous': parse_indices(row['[tensor_index-es for ambigious images]']),
        'incorrect': parse_indices(row['[tensor_index-es for incorrect images]']),
        'checked': int(row['checked'])
    }

def load_qwen_predictions(model, phrase, strategy):
    """Load Qwen predictions for model, phrase, and strategy."""
    qwen_dir = QWEN_BASE_DIR / model / strategy
    phrase_file_map = {
        'close by': 'closeby',
    }
    phrase_file = phrase_file_map.get(phrase, phrase.replace(' ', '_'))
    files = sorted(qwen_dir.glob(f"qwen_{phrase_file}_*.csv"))
    if not files:
        return pd.DataFrame()
    df = pd.read_csv(files[0])
    return df[df['phrase'] == phrase].copy()

def create_confusion_matrix(y_true, y_pred):
    """Create confusion matrix."""
    labels = ['Yes', 'No', 'Ambiguous']
    matrix = pd.DataFrame(0, index=labels, columns=labels)
    for t, p in zip(y_true, y_pred):
        if t in labels and p in labels:
            matrix.loc[t, p] += 1
    return matrix

## Show Available Phrases

Run this cell first to see what phrases are available for each model.

In [10]:
# print("Available phrases per model:\n")
# for model in MODELS:
#     df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
#     phrases = df['phrase'].dropna().unique().tolist()
#     print(f"{model.upper()}:")
#     for p in phrases:
#         print(f"  - {p}")
#     print()

## Run Analysis

**Set your model and phrase below:**

In [11]:
# ===== USER INPUT =====
MODEL = 'standard'     # Choose: 'redcaps', 'cogvlm', or 'standard'
PHRASE = 'to the left of'      # Type the phrase (see available phrases above)
# ======================

print(f"Analyzing: {MODEL} / {PHRASE}\n")

# Load manual annotations
manual_data = load_manual_annotations(MODEL, PHRASE)
if manual_data is None:
    print(f"ERROR: No manual annotations found for {MODEL}/{PHRASE}")
else:
    print(f"Manual annotations loaded: {manual_data['checked']} images checked\n")
    
    # Process each strategy
    for strategy in STRATEGIES:
        print(f"{'='*70}")
        print(f"STRATEGY: {strategy.upper()}")
        print(f"{'='*70}\n")
        
        # Load Qwen predictions
        qwen_df = load_qwen_predictions(MODEL, PHRASE, strategy)
        
        if qwen_df.empty:
            print(f"No Qwen predictions found for {strategy}\n")
            continue
        
        # Compare predictions
        results = []
        y_true, y_pred = [], []
        
        for _, row in qwen_df.iterrows():
            idx = int(row['tensor_index'])
            qwen_ans = get_qwen_answer(row)
            manual_ans = get_manual_label(idx, manual_data)
            
            if manual_ans:
                y_true.append(manual_ans)
                y_pred.append(qwen_ans)
                
                results.append({
                    'tensor_index': idx,
                    'image': f"{manual_data['tensor_file']}[{idx}]",
                    'image_prompt': f"{row.get('class1', '')} {PHRASE} {row.get('class2', '')}".strip(),
                    'manual_annotation': manual_ans,
                    'qwen_answer': qwen_ans,
                    'qwen_full_answer': row.get('full_output', ''),
                    'qwen_prompt_type': strategy,
                    'match': manual_ans == qwen_ans
                })
        
        results_df = pd.DataFrame(results)
        
        # 1. Confusion Matrix
        if y_true:
            conf_matrix = create_confusion_matrix(y_true, y_pred)
            print("CONFUSION MATRIX (Rows=Manual, Cols=Qwen):")
            print(conf_matrix)
            print()
        
        # 2. Accuracy
        if y_true:
            accuracy = sum([t == p for t, p in zip(y_true, y_pred)]) / len(y_true) * 100
            print(f"ACCURACY: {accuracy:.2f}% ({sum([t == p for t, p in zip(y_true, y_pred)])}/{len(y_true)})\n")
        
        # 3. Mismatch Breakdown
        mismatches = results_df[~results_df['match']]
        if not mismatches.empty:
            print(f"MISMATCHES ({len(mismatches)} total):")
            display(mismatches[['image', 'image_prompt', 'manual_annotation', 'qwen_answer', 
                               'qwen_full_answer', 'qwen_prompt_type']])
        else:
            print("No mismatches!\n")
        
        # 4. All Predictions Summary
        all_qwen = qwen_df.copy()
        all_qwen['qwen_normalized'] = all_qwen.apply(get_qwen_answer, axis=1)
        counts = all_qwen['qwen_normalized'].value_counts()
        total = len(all_qwen)
        print(f"\nALL QWEN PREDICTIONS (Total: {total}):")
        for label in ['Yes', 'No', 'Ambiguous']:
            cnt = counts.get(label, 0)
            print(f"  {label}: {cnt} ({cnt/total*100:.1f}%)")
        print("\n")

Analyzing: standard / to the left of

Manual annotations loaded: 100 images checked

STRATEGY: DIRECT

CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         39  14          0
No           2  36          0
Ambiguous    0   8          0

ACCURACY: 75.76% (75/99)

MISMATCHES (24 total):


,image,image_prompt,manual_annotation,qwen_answer,qwen_full_answer,qwen_prompt_type
6,phrase_46534700.pt[7],oven to the left of vase,No,Yes,Yes,direct
10,phrase_46534700.pt[11],baseball bat to the left of dog,Yes,No,No,direct
11,phrase_46534700.pt[12],orange to the left of traffic light,Yes,No,No.,direct
15,phrase_46534700.pt[16],sheep to the left of computer mouse,Yes,No,No,direct
17,phrase_46534700.pt[18],knife to the left of umbrella,Ambiguous,No,No,direct
18,phrase_46534700.pt[19],motorcycle to the left of carrot,Ambiguous,No,No,direct
21,phrase_46534700.pt[22],carrot to the left of truck,Ambiguous,No,No,direct
29,phrase_46534700.pt[30],bicycle to the left of bus,Ambiguous,No,No,direct
31,phrase_46534700.pt[32],frisbee to the left of cat,Yes,No,No,direct
33,phrase_46534700.pt[34],toilet to the left of cat,Yes,No,No,direct



ALL QWEN PREDICTIONS (Total: 100):
  Yes: 42 (42.0%)
  No: 58 (58.0%)
  Ambiguous: 0 (0.0%)


STRATEGY: EXPERT

CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         51   2          0
No           8  28          2
Ambiguous    3   5          0

ACCURACY: 79.80% (79/99)

MISMATCHES (20 total):


,image,image_prompt,manual_annotation,qwen_answer,qwen_full_answer,qwen_prompt_type
7,phrase_46534700.pt[8],bed to the left of book,No,Yes,"1. The bed is the surface covered with white sheets and pillows, occupying the lower and central portion of the image. It extends horizontally across the frame, with the book resting on it. 2. The book is a red hardcover book with gold lettering on the spine. It is standing upright on the bed, positioned in the center of the image, leaning against the white pillows. 3. Based on their relative positions, the book is centered on the bed. Since the bed extends to the left and right of the book, the bed is not entirely to the left of the book. However, the bed does extend to the left of the book's position. Conclusion: Yes",expert
11,phrase_46534700.pt[12],orange to the left of traffic light,Yes,No,"1. The orange is the illuminated light within the traffic light assembly. It is located in the top circular section of the traffic light. 2. The traffic light is a vertical unit with two lights (top and bottom) housed in a yellow casing. It is mounted on a horizontal green pole, which is attached to a vertical green pole. 3. The orange light is in the top section of the traffic light, which is positioned to the right of the vertical pole. The entire traffic light structure is on the right side of the frame relative to the pole. The orange light is not a separate object to the left of the traffic light; it is part of the traffic light itself. Therefore, the orange is not to the left of the traffic light—it is part of it. Conclusion: No",expert
17,phrase_46534700.pt[18],knife to the left of umbrella,Ambiguous,No,"1. The knife is positioned vertically in the center of the image, aligned with the central pole of the umbrella. Its blade is oriented upward, and the handle extends downward, resting along the central axis. 2. The umbrella is fully open and fills the entire frame of the image. It is viewed from directly underneath, with its canopy radiating outward from the central pole. 3. The knife is located directly on the central axis of the umbrella, which is the vertical centerline of the image. Since the umbrella's structure is symmetrical around this axis, the knife is not to the left or right of the umbrella but rather aligned with its center. Conclusion: No",expert
18,phrase_46534700.pt[19],motorcycle to the left of carrot,Ambiguous,Yes,"1. The motorcycle is a dirt bike with an orange and white color scheme, positioned horizontally in the image, oriented from left to right. It is located in the upper-left to central portion of the frame, with its front wheel on the right and rear wheel on the left. 2. The carrot is a long, orange vegetable with a green leafy top, lying horizontally on the ground. It is positioned in the lower-right portion of the image, to the right of the motorcycle's rear wheel. 3. Based on their relative positions, the motorcycle extends from the left side of the image toward the center, while the carrot is located further to the right, near the center-right. Therefore, the motorcycle is to the left of the carrot. Conclusion: Yes",expert
21,phrase_46534700.pt[22],carrot to the left of truck,Ambiguous,No,"1. The carrot is a large, orange carrot with green leafy stems, positioned horizontally on the flatbed of the truck. 2. The truck is a large vehicle with a black cab and a flatbed trailer, visible from the side. 3. The carrot is located on the flatbed of the truck, extending from the left side towards the center of the frame. The truck's cab is on the left, and the flatbed extends to the right. From the viewer's perspective, the carrot is positioned to the right of the truck's cab and appears to be on the same horizontal level as the truck's flatbed. Conclusion: No",expert
29,phrase_46534700.pt[30],bicycle to the left of bus,Ambiguous,Yes,"1. The bicycle is a white bicycle with a black seat and handlebars, positioned in the lower central part of the image, leaning against the 


ALL QWEN PREDICTIONS (Total: 100):
  Yes: 63 (63.0%)
  No: 35 (35.0%)
  Ambiguous: 2 (2.0%)


STRATEGY: INVERSE

CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         48   0          5
No           8  11         19
Ambiguous    4   1          3

ACCURACY: 62.63% (62/99)

MISMATCHES (37 total):


,image,image_prompt,manual_annotation,qwen_answer,qwen_full_answer,qwen_prompt_type
4,phrase_46534700.pt[5],computer keyboard to the left of skateboard,No,Ambiguous,"Let's analyze the image and the logical statements step by step. 1. **Is the skateboard to the right of the computer keyboard?** - Looking at the image: The skateboard is on the **left side** of the frame, and the computer keyboard is also on the **left side**, but **to the right of the skateboard**. - So, the skateboard is **to the left** of the keyboard. - Therefore, the skateboard is **not** to the right of the keyboard. - **Answer to question 1: No** 2. **If so, then the computer keyboard is to the left of the skateboard.** - This is a conditional statement: *If* the skateboard is to the right of the keyboard, *then* the keyboard is to the left of the skateboard. - But the ""if"" condition is **false** (as we determined in step 1). - In logic, if the premise is false, the conditional statement is **not** invalidated — it's considered **true by default** in classical logic. However, the conclusion is not necessarily true or false based on a false premise. - But here, the question is asking us to evaluate the **conclusion** based",inverse
5,phrase_46534700.pt[6],toilet to the left of potted plant,No,Ambiguous,"Let's analyze the image and the logical structure of the question. 1. **Is the potted plant to the right of the toilet?** - Looking at the image: The potted plant is on the **left** side of the frame, and the toilet is on the **right** side. - Therefore, the potted plant is **not** to the right of the toilet. - So the answer to question 1 is **No**. 2. **If so, then the toilet is to the left of the potted plant.** - This is a conditional statement: *If* the plant is to the right of the toilet, *then* the toilet is to the left of the plant. - But the condition (""the plant is to the right of the toilet"") is **false**, so the conditional does not apply. - However, the conclusion is asking us to evaluate the **logical consistency** of the reasoning based on the spatial relationship. Now, the **conclusion** is: > [Yes/No/Ambiguous] We are to determine whether the **conclusion** (i.e., the second statement) is valid based on the first. But the second statement is",inverse
6,phrase_46534700.pt[7],oven to the left of vase,No,Ambiguous,"Let's analyze the image and the logical statements step by step. ### Step 1: Determine the relative positions of the vase and the oven. - The image shows a black oven in the center. - To the **left** of the oven, there is a **black vase**. - To the **right** of the oven, there is a **white vase** containing wooden utensils. So, there are **two vases** in the image: - One black vase on the **left**. - One white vase on the **right**. The question is: **""Is the vase to the right of the oven?""** Since there is a **white vase** on the **right** of the oven, the answer to this question is **Yes**. ### Step 2: Evaluate the conditional statement. The second statement says: **""If so, then the oven is to the left of the vase.""** We've established that **the vase (the white one) is to the right of the oven**, so the condition ""if so"" is true. Therefore, the conclusion should be: **The oven is to the left of the vase.** This is logically correct, because if the vase is to the right of the",inverse
11,phrase_46534700.pt[12],orange to the left of traffic light,Yes,Ambiguous,"Let's analyze the image and the logical structure of the question. ### Step 1: Identify the objects - The image shows a traffic light. - The traffic light has two visible lights: the top one is **red**, and the bottom one is **green**. - The **orange** light (also known as amber) is **not lit** — it is the middle light, which is currently off. - The **traffic light** is a single unit, mounted on a pole. ### Step 2: Address the first question > **1. Is the traffic light to the right of the orange?** - The **orange** refers to the **orange light** (the middle light in 


ALL QWEN PREDICTIONS (Total: 100):
  Yes: 61 (61.0%)
  No: 12 (12.0%)
  Ambiguous: 27 (27.0%)




In [12]:
OUTPUT_DIR = MANUAL_DIR
LABELS = ['Yes', 'No', 'Ambiguous']
STRATEGY_ORDER = {name: index for index, name in enumerate(STRATEGIES)}
LABEL_ORDER = {name: index for index, name in enumerate(LABELS)}


def export_confusion_matrices_for_model(model):
    """Export one readable CSV per model with all phrase/strategy confusion matrices."""
    manual_df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
    export_rows = []
    summary_rows = []

    for _, manual_row in manual_df.iterrows():
        phrase = manual_row['phrase']
        manual_data = load_manual_annotations(model, phrase)
        if manual_data is None:
            continue

        for strategy in STRATEGIES:
            qwen_df = load_qwen_predictions(model, phrase, strategy)
            if qwen_df.empty:
                continue

            y_true, y_pred = [], []
            for _, row in qwen_df.iterrows():
                idx = int(row['tensor_index'])
                manual_ans = get_manual_label(idx, manual_data)
                if manual_ans is None:
                    continue
                y_true.append(manual_ans)
                y_pred.append(get_qwen_answer(row))

            if not y_true:
                continue

            conf_matrix = create_confusion_matrix(y_true, y_pred)
            total_compared = len(y_true)
            correct_predictions = int(sum(t == p for t, p in zip(y_true, y_pred)))
            accuracy_pct = round(correct_predictions / total_compared * 100, 2)

            for manual_label in LABELS:
                export_rows.append({
                    'phrase': phrase,
                    'strategy': strategy,
                    'manual_label': manual_label,
                    'qwen_yes': int(conf_matrix.loc[manual_label, 'Yes']),
                    'qwen_no': int(conf_matrix.loc[manual_label, 'No']),
                    'qwen_ambiguous': int(conf_matrix.loc[manual_label, 'Ambiguous']),
                    'row_total': int(conf_matrix.loc[manual_label].sum()),
                    'total_compared': total_compared,
                    'correct_predictions': correct_predictions,
                    'accuracy_pct': accuracy_pct,
                    'tensor_file': manual_data['tensor_file'],
                })

            summary_rows.append({
                'model': model,
                'phrase': phrase,
                'strategy': strategy,
                'total_compared': total_compared,
                'correct_predictions': correct_predictions,
                'accuracy_pct': accuracy_pct,
            })

    export_df = pd.DataFrame(export_rows)
    if export_df.empty:
        return None, pd.DataFrame()

    export_df['_strategy_order'] = export_df['strategy'].map(STRATEGY_ORDER)
    export_df['_label_order'] = export_df['manual_label'].map(LABEL_ORDER)
    export_df = export_df.sort_values(
        by=['phrase', '_strategy_order', '_label_order'],
        kind='stable'
    ).drop(columns=['_strategy_order', '_label_order'])

    output_path = OUTPUT_DIR / f'confusion_matrices_{model}.csv'
    export_df.to_csv(output_path, index=False)

    return output_path, pd.DataFrame(summary_rows)


created_files = []
summary_frames = []

for model in MODELS:
    output_path, summary_df = export_confusion_matrices_for_model(model)
    if output_path is not None:
        created_files.append(str(output_path))
        summary_frames.append(summary_df)

print('Created files:')
for path in created_files:
    print(f'  - {path}')

if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_df = summary_df.sort_values(by=['model', 'phrase', 'strategy'], kind='stable')
    # print('\nExport summary:')
    # display(summary_df)
else:
    print('No confusion matrices were exported.')

Created files:
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_redcaps.csv
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_cogvlm.csv
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_standard.csv


## Run Analysis for Old Combined CSV Files

Use this when an old CSV already contains all strategies (`direct`, `expert`, `inverse`) in one file with columns like `sd_model_type,tensor_file,tensor_index,phrase,strategy,full_output,predicted_label,class1,class2`.

In [13]:
# # ===== USER INPUT (OLD COMBINED CSV) =====
# MODEL_OLD = 'cogvlm'  # Choose: 'redcaps', 'cogvlm', or 'standard'
# PHRASE_OLD = 'between'  # Type the phrase
# OLD_CSV_PATH = BASE_DIR / 'x7-z1-qwen_small/new_qwen_outputs/qwen_cogvlm_between_inverse_expert_direct_20260303_1919.csv'
# # ==========================================

# print(f"Analyzing old combined CSV: {OLD_CSV_PATH}")
# print(f"Target: {MODEL_OLD} / {PHRASE_OLD}\n")

# manual_data_old = load_manual_annotations(MODEL_OLD, PHRASE_OLD)
# if manual_data_old is None:
#     print(f"ERROR: No manual annotations found for {MODEL_OLD}/{PHRASE_OLD}")
# elif not OLD_CSV_PATH.exists():
#     print(f"ERROR: CSV not found: {OLD_CSV_PATH}")
# else:
#     old_df = pd.read_csv(OLD_CSV_PATH)

#     # Keep only rows for selected model/phrase to match the new analysis behavior.
#     old_df = old_df[
#         (old_df['phrase'] == PHRASE_OLD)
#         & (old_df['sd_model_type'].str.lower() == MODEL_OLD.lower())
#     ].copy()

#     if old_df.empty:
#         print("No rows found in old CSV for the selected model/phrase.")
#     else:
#         available_strategies = [s for s in STRATEGIES if s in set(old_df['strategy'].dropna().str.lower())]
#         print(f"Manual annotations loaded: {manual_data_old['checked']} images checked")
#         print(f"Found strategies in CSV: {available_strategies}\n")

#         for strategy in available_strategies:
#             print(f"{'='*70}")
#             print(f"STRATEGY: {strategy.upper()}")
#             print(f"{'='*70}\n")

#             qwen_df_old = old_df[old_df['strategy'].str.lower() == strategy].copy()

#             if qwen_df_old.empty:
#                 print(f"No predictions found for {strategy}\n")
#                 continue

#             results = []
#             y_true, y_pred = [], []

#             for _, row in qwen_df_old.iterrows():
#                 idx = int(row['tensor_index'])
#                 qwen_ans = get_qwen_answer(row)
#                 manual_ans = get_manual_label(idx, manual_data_old)

#                 if manual_ans:
#                     y_true.append(manual_ans)
#                     y_pred.append(qwen_ans)

#                     results.append({
#                         'tensor_index': idx,
#                         'image': f"{manual_data_old['tensor_file']}[{idx}]",
#                         'image_prompt': f"{row.get('class1', '')} {PHRASE_OLD} {row.get('class2', '')}".strip(),
#                         'manual_annotation': manual_ans,
#                         'qwen_answer': qwen_ans,
#                         'qwen_full_answer': row.get('full_output', ''),
#                         'qwen_prompt_type': strategy,
#                         'match': manual_ans == qwen_ans
#                     })

#             results_df = pd.DataFrame(results)

#             # 1. Confusion Matrix
#             if y_true:
#                 conf_matrix = create_confusion_matrix(y_true, y_pred)
#                 print("CONFUSION MATRIX (Rows=Manual, Cols=Qwen):")
#                 print(conf_matrix)
#                 print()

#             # 2. Accuracy
#             if y_true:
#                 correct = sum([t == p for t, p in zip(y_true, y_pred)])
#                 accuracy = correct / len(y_true) * 100
#                 print(f"ACCURACY: {accuracy:.2f}% ({correct}/{len(y_true)})\n")

#             # 3. Mismatch Breakdown
#             mismatches = results_df[~results_df['match']]
#             if not mismatches.empty:
#                 print(f"MISMATCHES ({len(mismatches)} total):")
#                 display(mismatches[['image', 'image_prompt', 'manual_annotation', 'qwen_answer',
#                                    'qwen_full_answer', 'qwen_prompt_type']])
#             else:
#                 print("No mismatches!\n")

#             # 4. All Predictions Summary
#             all_qwen = qwen_df_old.copy()
#             all_qwen['qwen_normalized'] = all_qwen.apply(get_qwen_answer, axis=1)
#             counts = all_qwen['qwen_normalized'].value_counts()
#             total = len(all_qwen)
#             print(f"\nALL QWEN PREDICTIONS (Total: {total}):")
#             for label in ['Yes', 'No', 'Ambiguous']:
#                 cnt = counts.get(label, 0)
#                 print(f"  {label}: {cnt} ({cnt/total*100:.1f}%)")
#             print("\n")